In [0]:
%run ../../00_common/data_utils

In [0]:
def backup_cbr_datasets(task_id, retention_days=30):
    """
    备份本批次将要删除的 CBR dataset 数据到 t_cbr_dataset_backup。
    根据 log 表中当前 task_id 的 anonymization keys (MarketCode + New_UniversalKey)
    匹配 CBR 表 (MarketCode + MDMKey)，确保备份与后续删除的数据一致。
    源表:
      - t_cbr_dataset          -> cbr_type = 'cbr'
      - t_cbr_withoutpii_dataset -> cbr_type = 'cbrpublic'
      - t_cbrdrjart_dataset    -> cbr_type = 'cbrdj'
    """
    combine_db = get_env_config('golden_consumer_combine_database')
    target_db = get_env_config('silver_mdm_anonymization_database')

    backup_table = f"{target_db}.t_cbr_dataset_backup"
    log_table = f"{target_db}.t_mdm_anonymization_log"

    log_keys = (
        spark.table(log_table)
        .where(
            (F.col("status") == ANON_STATUS_IN_PROGRESS) &
            (F.col("task_id") == task_id)
        )
        .select(
            F.col("MarketCode"),
            F.col("New_UniversalKey").alias("MDMKey")
        )
        .distinct()
    )

    log_keys = log_keys.cache()
    try:
        if log_keys.isEmpty():
            print("No anonymization keys for this task, skip backup")
            return

        cbr_sources = [
            (f"{combine_db}.t_cbr_dataset", ANON_CBR_TYPE_CBR),
            (f"{combine_db}.t_cbr_withoutpii_dataset", ANON_CBR_TYPE_CBRPUBLIC),
            (f"{combine_db}.t_cbrdrjart_dataset", ANON_CBR_TYPE_CBRDJ),
        ]

        backup_parts = []
        for table_name, cbr_type in cbr_sources:
            backup_parts.append(
                spark.table(table_name)
                     .join(F.broadcast(log_keys), ["MarketCode", "MDMKey"], "left_semi")
                     .select(
                         F.col("MarketCode"),
                         F.col("MDMKey"),
                         F.col("FinalJSON"),
                         F.lit(cbr_type).alias("cbr_type"),
                         F.current_timestamp().alias("create_time"),
                         F.expr(f"current_timestamp() + INTERVAL {retention_days} DAYS").alias("expired_time"),
                         F.lit(task_id).alias("task_id"),
                     )
            )

        backup_df = backup_parts[0]
        for part in backup_parts[1:]:
            backup_df = backup_df.unionAll(part)

        record_count = backup_df.count()
        if record_count == 0:
            print("No CBR dataset records to backup")
            return

        save_to_target_table(backup_df, backup_table, f"task_id='{task_id}'")
        print(f"Backup completed: {record_count} records written to {backup_table}")
    finally:
        log_keys.unpersist()

In [0]:
task_id = get_ex_param("task_id", "")
retention_days = int(get_ex_param("retention_days", "30"))
print(f"task_id: {task_id}, backup records retention_days: {retention_days}")

step_name = "backup_cbr_datasets"
step_num = "02"
project = "dataanonymization"
log_table_name = f"{get_env_config('config_database')}.t_task_step_log"

start_time = datetime.now()
status = "SUCCESS"
message = "completed"

try:
    backup_cbr_datasets(task_id, retention_days)
except Exception as e:
    status = "FAILED"
    message = f"{type(e).__name__}: {str(e)}"
    raise
finally:
    end_time = datetime.now()
    append_step_log(
        log_table_name=log_table_name,
        task_id=task_id,
        step_num=step_num,
        step_name=step_name,
        start_time=start_time,
        end_time=end_time,
        status=status,
        message=message,
        project=project
    )